In [1]:
dataset = []
with open('CAR FINANCING AGREEMENT.txt', 'r', encoding='utf-8') as file:
    dataset = file.readlines()
    print(f'Loaded {len(dataset)} entries')

Loaded 17 entries


In [ ]:
for e in dataset:
    print(e.strip())

BORROWER: Carlos Brown, residing at 1500 Brazil Avenue - Boston, MA, holder of SSN nº 360132173.
LENDER: FINANCIAL BANK OF AMERICA Inc., registered under EIN nº 00-0000000, headquartered at 1000 Main Avenue, New York, NY.
FINANCED AMOUNT: $82,437.00.
CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).
CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the financed asset, with minimum coverage as required by current legislation.
CLAUSE SIX – DEFAULT: Payment delay will result in a penal

In [3]:
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))


In [4]:
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')


Added chunk 1/17 to the database
Added chunk 2/17 to the database
Added chunk 3/17 to the database
Added chunk 4/17 to the database
Added chunk 5/17 to the database
Added chunk 6/17 to the database
Added chunk 7/17 to the database
Added chunk 8/17 to the database
Added chunk 9/17 to the database
Added chunk 10/17 to the database
Added chunk 11/17 to the database
Added chunk 12/17 to the database
Added chunk 13/17 to the database
Added chunk 14/17 to the database
Added chunk 15/17 to the database
Added chunk 16/17 to the database
Added chunk 17/17 to the database


In [5]:
import numpy as np
#run once after the VectorDB is created
chunks = [chunk for chunk, embed in VECTOR_DB]

In [6]:
chunks[:5]

['BORROWER: Carlos Brown, residing at 1500 Brazil Avenue - Boston, MA, holder of SSN nº 360132173.\n',
 'LENDER: FINANCIAL BANK OF AMERICA Inc., registered under EIN nº 00-0000000, headquartered at 1000 Main Avenue, New York, NY.\n',
 'FINANCED AMOUNT: $82,437.00.\n',
 'CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.\n',
 'CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.\n']

In [7]:
#convert vectorDB to a numpy array
chunk_vectors = np.asarray(
    [embed for chunk, embed in VECTOR_DB], dtype=np.float32
)
#normalize the vectors choose the maximum between the norm and a small value to avoid division by zero
chunk_vectors /= np.maximum(
    np.linalg.norm(chunk_vectors, axis = 1, keepdims = True), 1e-10
)

In [8]:
chunk_vectors.shape

(17, 768)

In [9]:
def retrieve(query, top_n = 5):
        query_vectors = np.asarray(ollama.embed(model =EMBEDDING_MODEL, input = query)['embeddings'][0])
        query_vectors /= max(
                    np.linalg.norm(query_vectors), 1e-10
                    )
        similarities = chunk_vectors @ query_vectors
        top_indices = np.argsort(similarities)[-top_n:][::-1]
        return [(chunks[i], float(similarities[i])) for i in top_indices]

In [47]:
input_query = input('Ask me a question: ')# type your question here

In [48]:
input_query

'According to the provided contract text, what payment amounts and due dates does the schedule specify?Quote or summarize only what the document says.'

In [49]:
retrieved_knowledge = retrieve(input_query)

In [50]:
retrieved_knowledge

[('CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.\n',
  0.6306227782857634),
 ('CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).\n',
  0.6204888263633438),
 ('CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.\n',
  0.6158265062604729),
 ('CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.\n',
  0.6125689791323872),
 ('CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.\n',
  0.6107240023382352)]

In [51]:
print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

Retrieved knowledge:
 - (similarity: 0.63) CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.

 - (similarity: 0.62) CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).

 - (similarity: 0.62) CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.

 - (similarity: 0.61) CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.

 - (similarity: 0.61) CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.



In [52]:
context = '\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{context}
'''

In [53]:
print(context)

 - CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.

 - CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).

 - CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.

 - CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.

 - CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.



In [54]:
print(instruction_prompt)

You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
 - CLAUSE SIX – DEFAULT: Payment delay will result in a penalty of 2% on the installment, late interest of 1% per month, and possible execution of the guarantee.

 - CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).

 - CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the acquisition of the asset described in the preamble.

 - CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed upon by the parties.

 - CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of the BANK until full settlement.




In [55]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)


Chatbot response:
The schedule specifies:

* Payment delay: 2% on the installment
* Late interest: 1% per month
* Annual correction of payments: corrected annually by the Consumer Price Index (CPI)